# Udaplay Project

Based on UDAPLAY Starter Project


## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

Import remaining libraries
Some cleanup for later: there might me some imports whithin the code which I later have to move here

In [2]:
# First import al OS related libarries and read the .env 
# check if all necessary variables are set!

import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [3]:
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [4]:
# TODO
# Create a .env file in this folder (project/starter) with the following variables
# (copy .env.example from the repo root):
# OPENAI_API_KEY="voc-..."
# OPENAI_BASE_URL="https://openai.vocareum.com/v1"
# TAVILY_API_KEY="tvly-..."

# is defined ad loaded aready before

### VectorDB Instance

In [5]:
# TODO: Instantiate your ChromaDB Client
# Keep this path: Part 2 loads the database from "chromadb" (inside project/starter)

chroma_client = chromadb.PersistentClient(path="chromadb")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


### Collection

In [6]:
# TODO: Pick one embedding function
# Part 2 rebuilds this same function when it loads the collection,
# so if you pick something different, keep it identical in both notebooks.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
     api_key_env_var="OPENAI_API_KEY",
     api_base=os.getenv("OPENAI_BASE_URL"),
 )

In [7]:
# TODO: Create a collection
# Keep this name: Part 2 loads the collection "udaplay"
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Add documents
Read files from disk in  JSON format

In [8]:
# Run this notebook from project/starter: the game files are in ./games
data_dir = "games"
num_docs = 0

# Check if data_dir exist in fielsystem 
if not os.path.exists(data_dir):
    print(f"Directory '{data_dir}' not found!")
    print("Data Directory does not exist!!!")
else:
    # Read the files (must b in JSON format and ending must be .json)
    json_files = [f for f in os.listdir(data_dir) if f.endswith('.json')]
    print(f"Found {len(json_files)} JSON files in '{data_dir}' directory")
    
    # files loaded
    print(f"-->  {len(json_files)} files loaded")





Found 15 JSON files in 'games' directory
-->  15 files loaded


### Loading ChromaDB
Read again from disk in JSON format and store it directly to ChromaDB

In [9]:
# Add documents to ChromaDB
num_docs = 0
    
for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue
            
    file_path = os.path.join(data_dir, file_name)
        
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            game = json.load(f)
            
        # Create content string for embedding
        content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"
            
        # Use filename (without extension) as document ID
        doc_id = os.path.splitext(file_name)[0]
            
        # Add to collection
        collection.add(
            ids=[doc_id],
            documents=[content],
            metadatas=[game]
        )
            
        num_docs = num_docs + 1        
    except Exception as e:
            print(f"Error processing {file_name}: {e}")
            
print(f"Added {num_docs} documents to ChromaDB of {collection.count()} documemts from collection")



Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Added 15 documents to ChromaDB of 15 documemts from collection


### Demonstration of Semantic Search
Show that semantic search works properly with data previously read from disk and stored to ChromDB

In [10]:
# Testfunction "Semantic Search"

def semanticsearch_test(collection, query, n_results=3):
    """Test semantic search and display results"""
    print(f"\n--> Semantic Search of: '{query}' <--")
    
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=['documents', 'metadatas', 'distances']
    )
    
    if results['documents'][0]:
        print(f"Found {len(results['documents'][0])} results:")
        
        for i, (doc, metadata, distance) in enumerate(zip(
            results['documents'][0], 
            results['metadatas'][0], 
            results['distances'][0]
        )):
            similarity = 1 - distance  # DISTANCE --> SIMILARITY conversion
            print(f"\n{i+1}. {metadata['Name']} ({metadata['YearOfRelease']})")
            print(f"   Platform: {metadata['Platform']}")
            print(f"   Genre: {metadata.get('Genre', 'N/A')}")
            print(f"   Publisher: {metadata.get('Publisher', 'N/A')}")
            print(f"   Similarity: {similarity:.3f}")
            print(f"   Description: {metadata['Description'][:100]}...")
    else:
        print("Nothing found!")
    
    return results

# Sample queries
test_queries = [
    "Family suitable games",
    "Nintendo jump and run games",
    "PlayStation 1 games from the 90s",
    "RPGs with fantasy themes",
    "Action games with shooting mechanics"
]

# execute the test search and limit the results to 2 per questions --> results need to be viewed in editor
for query in test_queries:
    semanticsearch_test(collection, query, n_results=3)


--> Semantic Search of: 'Family suitable games' <--


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Found 3 results:

1. Wii Sports (2006)
   Platform: Wii
   Genre: Sports
   Publisher: Nintendo
   Similarity: 0.574
   Description: A collection of sports games that utilize the Wii's motion controls, bundled with the console to sho...

2. Kinect Adventures! (2010)
   Platform: Xbox 360
   Genre: Party
   Publisher: Microsoft Game Studios
   Similarity: 0.563
   Description: A collection of mini-games designed to showcase the capabilities of the Kinect motion sensor....

3. Gran Turismo 5 (2010)
   Platform: PlayStation 3
   Genre: Racing
   Publisher: Sony Computer Entertainment
   Similarity: 0.550
   Description: A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic d...

--> Semantic Search of: 'Nintendo jump and run games' <--
Found 3 results:

1. Super Mario 64 (1996)
   Platform: Nintendo 64
   Genre: Platformer
   Publisher: Nintendo
   Similarity: 0.650
   Description: A groundbreaking 3D platformer that set new standards for the g

In [11]:
# show collection statistics
print("--> ChromaDB collection statistics <--")
print(f"Name of Collection  : {collection.name}")
print(f"Number of  Documents: {collection.count()}")

# Get 4 document samples to verify data structure
document_samples = collection.get(limit=4, include=['documents', 'metadatas'])

print("\n--> Document Samples <-- ")
for i, (doc, metadata) in enumerate(zip(document_samples['documents'], document_samples['metadatas'])):
    print(f"\n{i+1}. Document ID: {document_samples['ids'][i]}")
    print(f"   Game: {metadata['Name']}")
    print(f"   Platform: {metadata['Platform']}")
    print(f"   Year: {metadata['YearOfRelease']}")
    print(f"   Content: {doc[:100]}...")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


--> ChromaDB collection statistics <--
Name of Collection  : udaplay
Number of  Documents: 15

--> Document Samples <-- 

1. Document ID: 001
   Game: Gran Turismo
   Platform: PlayStation 1
   Year: 1997
   Content: [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars an...

2. Document ID: 002
   Game: Grand Theft Auto: San Andreas
   Platform: PlayStation 2
   Year: 2004
   Content: [PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the ficti...

3. Document ID: 003
   Game: Gran Turismo 5
   Platform: PlayStation 3
   Year: 2010
   Content: [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection ...

4. Document ID: 004
   Game: Marvel's Spider-Man
   Platform: PlayStation 4
   Year: 2018
   Content: [PlayStation 4] Marvel's Spider-Man (2018) - An open-world superhero game that lets players swing th...


### Completed Tasks:

- ChromaDB vector database setup with persistent storage
- Set up OpenAI embeddings for semantic search
- Loaded game JSON files from disk
- Added all games to ChromaDB with metadata
- verification of semantic search
